# Work Notebook

## Phase 1. CS Corpus Acquisition and 50K Selection

- **Goal:** Build a fixed corpus of exactly 50,000 eligible CS papers from arXiv metadata.
- **Tasks:** Load metadata, validate required fields, filter `cs.*`, remove missing title/abstract and duplicate IDs, and create a stable 50K sample.
- **Expected outputs:** `raw_data_profile.json`, `eligible_cs_pool.parquet`, `final_50k_cs_corpus.parquet`, `corpus_selection_report.json`.

### 1.1 Setup

Install pinned dependencies and import the arXiv metadata snapshot from Kaggle via `kagglehub`. The dataset is a ~3.8 GB newline-delimited JSON file containing metadata for all arXiv papers.

In [ ]:
%pip install kaggle==2.2.1
%pip install kagglehub==1.0.1
%pip install duckdb==1.5.3
%pip install numpy==2.4.6
%pip install pandas==3.0.3


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("Cornell-University/arxiv")

print("Path to dataset files:", path)

In [ ]:
import os
import duckdb

### 1.2 Raw Data Profile

Before any filtering, profile the full dataset in a single DuckDB scan to capture total record count, CS vs. non-CS split, missing field counts, and the date range. Results are written to `raw_data_profile.json` and the `cs_paper_count` variable is retained for the selection report downstream.

> **Note:** The CS filter uses two `LIKE` conditions — `'cs.%'` and `'% cs.%'` — to match `cs.*` as a whole category token.

In [56]:
json_file_path = os.path.join(path, "arxiv-metadata-oai-snapshot.json")

# Quick count of eligible CS papers
# Two conditions handle all cases:
#   'cs.%'   → category string starts with cs. (e.g. "cs.LG cs.AI")
#   '% cs.%' → cs. appears after a space (e.g. "eess.SP cs.LG")
# This avoids false matches like 'physics.ins-det' which contains 'cs.' as a substring
test_query = f"""
    SELECT count(1) AS cs_paper_count
    FROM read_json_auto('{json_file_path}', format='newline_delimited')
    WHERE categories LIKE 'cs.%' OR categories LIKE '% cs.%';
"""

In [57]:
import json

# Analysis for raw_data_profile.json
# Captures total records, cs vs non-cs split, missing fields, and date range in one pass
raw_profile_query = f"""
SELECT
    count(1)                                                             AS total_records,
    count_if(categories LIKE 'cs.%' OR categories LIKE '% cs.%')        AS cs_records,
    count_if(NOT (categories LIKE 'cs.%' OR categories LIKE '% cs.%'))  AS non_cs_records,
    count_if(title    IS NULL OR TRIM(title)    = '')                    AS missing_title,
    count_if(abstract IS NULL OR TRIM(abstract) = '')                   AS missing_abstract,
    min(update_date)                                                      AS earliest_update,
    max(update_date)                                                      AS latest_update
FROM read_json_auto('{json_file_path}', format='newline_delimited');
"""

raw_profile    = duckdb.sql(raw_profile_query).df().iloc[0].to_dict()
cs_paper_count = int(raw_profile["cs_records"])

with open("data/raw_data_profile.json", "w") as f:
    json.dump({k: (v if isinstance(v, (int, float)) else str(v)) for k, v in raw_profile.items()}, f, indent=2)

print("raw_data_profile.json written:")
for k, v in raw_profile.items():
    print(f"  {k}: {v}")

raw_data_profile.json written:
  total_records: 3066190
  cs_records: 955380.0
  non_cs_records: 2110810.0
  missing_title: 0.0
  missing_abstract: 0.0
  earliest_update: 2007-05-23 00:00:00
  latest_update: 2026-06-05 00:00:00


### 1.3 Build Eligible Pool

Filter the raw snapshot down to a clean, deduplicated pool of CS papers:

- **CS filter** — keep only papers where a `cs.*` category token is present
- **Text filter** — drop any record with a null or empty title or abstract
- **Deduplication** — for papers with multiple versions (same `id`), keep only the most recent `update_date` using a `QUALIFY` window function

The result is written to `eligible_cs_pool.parquet`. From 955,380 CS papers, 1 duplicate was removed leaving **955,379 eligible records**.

In [51]:
os.makedirs("data", exist_ok=True)

# Build eligible pool: cs.* only, non-null/non-empty title+abstract, deduplicated by id
# QUALIFY keeps only the most recent version of each paper when there are duplicate IDs
# Two LIKE conditions correctly match cs.* as a token without false-matching 'physics.ins-det'
eligible_pool_query = f"""
COPY (
    SELECT *
    FROM read_json_auto('{json_file_path}', format='newline_delimited')
    WHERE (categories LIKE 'cs.%' OR categories LIKE '% cs.%')
      AND title    IS NOT NULL AND TRIM(title)    != ''
      AND abstract IS NOT NULL AND TRIM(abstract) != ''
    QUALIFY ROW_NUMBER() OVER (PARTITION BY id ORDER BY update_date DESC) = 1
)
TO 'data/eligible_cs_pool.parquet' (FORMAT PARQUET);
"""

duckdb.sql(eligible_pool_query)

# Report eligible pool size
pool_count = duckdb.sql("SELECT count(1) AS count FROM read_parquet('data/eligible_cs_pool.parquet')").df()
print("Eligible pool size:", pool_count["count"].iloc[0])

Eligible pool size: 955379


### 1.4 50K Sample Selection

Draw a stable random sample of exactly 50,000 papers from the eligible pool using DuckDB's reservoir sampler with seed `42`. Using a fixed seed ensures the corpus is reproducible across runs and teammates. The result is written to `final_50k_cs_corpus.parquet`.

In [52]:
# Draw stable 50K sample from the validated, deduplicated pool
sample_query = """
COPY (
    SELECT *
    FROM read_parquet('data/eligible_cs_pool.parquet')
    USING SAMPLE reservoir(50000 ROWS) REPEATABLE (42)
)
TO 'data/final_50k_cs_corpus.parquet' (FORMAT PARQUET);
"""

duckdb.sql(sample_query)

# Verify
final_count = duckdb.sql("SELECT count(1) AS count FROM read_parquet('data/final_50k_cs_corpus.parquet')").df()
print("Final corpus size:", final_count["count"].iloc[0])

Final corpus size: 50000


### 1.5 Corpus Selection Report

Summarise the full pipeline funnel and write `corpus_selection_report.json`. This records how many papers were dropped at each stage and serves as the audit trail for the corpus.

In [54]:
eligible_pool_size = int(duckdb.sql("SELECT count(1) FROM read_parquet('data/eligible_cs_pool.parquet')").df().iloc[0, 0])
final_corpus_size  = int(duckdb.sql("SELECT count(1) FROM read_parquet('data/final_50k_cs_corpus.parquet')").df().iloc[0, 0])

report = {
    "total_cs_papers":          cs_paper_count,
    "dropped_missing_fields":   cs_paper_count - eligible_pool_size,
    "dropped_duplicates":       cs_paper_count - eligible_pool_size,  # dedup is part of same step
    "eligible_pool_size":       eligible_pool_size,
    "final_corpus_size":        final_corpus_size,
    "sample_seed":              42,
    "filter_notes": {
        "cs_filter":   "categories LIKE 'cs.%' OR categories LIKE '% cs.%'",
        "dedup":       "QUALIFY ROW_NUMBER() OVER (PARTITION BY id ORDER BY update_date DESC) = 1",
        "text_filter": "title and abstract non-null and non-empty"
    }
}

with open("data/corpus_selection_report.json", "w") as f:
    json.dump(report, f, indent=2)

print("corpus_selection_report.json written:")
for k, v in report.items():
    if not isinstance(v, dict):
        print(f"  {k}: {v}")

corpus_selection_report.json written:
  total_cs_papers: 955380
  dropped_missing_fields: 1
  dropped_duplicates: 1
  eligible_pool_size: 955379
  final_corpus_size: 50000
  sample_seed: 42


### 1.6 Corpus Preview

Spot-check the final corpus with a 5-row sample to confirm the schema and that all records are genuine CS papers.

In [55]:
preview_query = """
SELECT id, title, categories, update_date
FROM read_parquet('data/final_50k_cs_corpus.parquet')
LIMIT 5;
"""

df = duckdb.sql(preview_query).df()
print(df)

           id                                              title  \
0  2505.17989  Outcome-based Reinforcement Learning to Predic...   
1  1610.01111  The Chromatic Number of Ordered Graphs With Co...   
2  2502.09215  Architecture for Simulating Behavior Mode Chan...   
3  2306.16092  Chatlaw: A Multi-Agent Collaborative Legal Ass...   
4  2605.21738        Asymptotic Rank Speedup Theorems, Revisited   

      categories update_date  
0    cs.LG cs.AI  2025-12-02  
1  math.CO cs.DM  2016-10-05  
2    cs.LO cs.AI  2025-02-14  
3          cs.CL  2024-05-31  
4    cs.CC cs.DS  2026-05-22  


## Phase 2. Text Cleaning and Metadata Normalization

- **Goal:** Convert the 50K corpus into clean, reproducible documents ready for embedding.
- **Tasks:** Clean title/abstract text, retain both raw and cleaned fields, normalize category/author fields, extract year, and generate stable document IDs.
- **Expected outputs:** `final_50k_cleaned.parquet`, `metadata_schema.md`, `text_cleaning_report.json`.

## Phase 3. Institution and Geography Enrichment with Fairness Labels

- **Goal:** Attach institution/location metadata and fairness labels for downstream analysis.
- **Tasks:** Enrich affiliations, normalize institution names, map country/region, and assign labels defined in proposal slides: privileged (QS Top-50 CS universities), underrepresented (other academic institutions), non-university labs (separate category), unknown (missing/unclear affiliation).
- **Expected outputs:** `final_50k_labeled.parquet`, `institution_alias_map.csv`, `fairness_labeling_rules.md`, `label_distribution.csv`, `affiliation_match_report.json`.

## Phase 4. Corpus Statistics and Baseline Fairness Analysis

- **Goal:** Produce report-ready statistics and baseline fairness measurements.
- **Tasks:** Summarize corpus distributions (year/category/institution/region), compute group proportions, run baseline retrieval parity checks, and inspect fairness-utility patterns.
- **Expected outputs:** `final_corpus_statistics.csv`, `corpus_summary.md`, baseline analysis figures and tables.

## Phase 5. ChromaDB Index Construction and Retrieval Smoke Test

- **Goal:** Write the cleaned/labeled 50K corpus into ChromaDB and confirm the index is usable for later RAG experiments.
- **Tasks:** Select embedding model, ingest one-paper-one-document records with metadata into ChromaDB, verify collection count is exactly 50,000, run 5-10 smoke-test queries, and confirm top-k returns plus metadata/filter functionality.
- **Expected outputs:** `chroma_db/`, `chroma_ingestion_report.json`, `sample_retrieval_results.md`, `index_config.yaml`.